# Mini practice 2 · Cohort QC: who does not belong?

**Genomic Data Processing · Day 2 (10/13) · about 15 minutes · practice slide 26**

You receive 2,000 exomes from a large cohort. Before anyone runs an association test, every
sample has to earn its place. Were we allowed to use it? Is it clean? Is it the person it claims
to be? Do we already have this person? Which analysis set does it belong to? You will apply the
rules Wang et al. (2021) used for the UK Biobank exomes, and keep a written reason for every
sample you remove.

> **Source of the rules.** Wang Q, Dhindsa RS, Carss K, *et al.* Rare variant contribution to
> human disease in 281,104 UK Biobank exomes. *Nature* 597, 527–532 (2021).
> doi:10.1038/s41586-021-03855-y

### What is real and what is synthetic
| Part | Status |
|---|---|
| QC rules: FREEMIX > 4%, < 94.5% of CCDS bases at ≥ 10×, KING kinship > 0.0884 (greedy), Pr(EUR) ≥ 0.99 and ±4 SD on PC1–PC4 | **Real**, from the Methods of Wang et al. 2021 |
| KING kinship categories (0.354 / 0.177 / 0.0884 / 0.0442) | **Real**, the KING guide |
| Sex-check and array-concordance cut-offs (0.20, 0.30, 0.95) | **Teaching thresholds** |
| All 2,000 samples, every metric, every kinship value | **Synthetic**. No UK Biobank data are used |

### How to work
Run the cells top to bottom; each takes about a second. **Predict, then run** where asked. Every
step says what it removes and how to check it. If a cell will not run, the verified tables are
in `checkpoints/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA  = Path("data/practice2")     # open this notebook from the day2-3-samples/ folder
CHECK = Path("checkpoints")
OUT   = Path("outputs")
OUT.mkdir(exist_ok=True)
pd.set_option("display.max_colwidth", 110)
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
                     "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
                     "legend.frameon": False, "axes.spines.top": False,
                     "axes.spines.right": False})

---
## 1 · Load and describe the cohort
*Concept slide: "Metadata: the sample's passport" (slide 7).*

Three files. The **manifest** says who each sample is supposed to be. The **QC metrics** say
what the sequencing data look like. The **kinship pairs** list every pair of samples that look
related (KING kinship > 0.0442). A *participant* is a person; a *sample* is one tube of their
DNA, and one person can have two samples.

In [ ]:
manifest = pd.read_csv(DATA / "cohort_manifest.tsv", sep="\t")
metrics  = pd.read_csv(DATA / "sample_qc_metrics.tsv", sep="\t")
kin      = pd.read_csv(DATA / "kinship_pairs.tsv", sep="\t")
print("manifest", manifest.shape, "| metrics", metrics.shape, "| kinship pairs", kin.shape)
print("samples:", manifest.sample_id.nunique(), "| participants:", manifest.participant_id.nunique())
manifest.head()

In [ ]:
print("batches :", manifest.sequencing_batch.value_counts().sort_index().to_dict())
print("centres :", manifest.recruitment_centre.value_counts().sort_index().to_dict())
print("specimen:", manifest.specimen.value_counts().to_dict())
print("sex     :", manifest.reported_sex.value_counts().to_dict())
print("age     :", manifest.age_at_recruitment.min(), "to", manifest.age_at_recruitment.max())
print("years   :", manifest.collection_year.min(), "to", manifest.collection_year.max())
print("consent :", manifest.consent_status.value_counts().to_dict())

### What each QC metric means
| column | in plain words | typical good value |
|---|---|---|
| `mean_depth` | how many reads cover an average targeted base | tens of reads |
| `pct_ccds_10x` | % of the consensus coding sequence (CCDS) covered by at least 10 reads: can we see every coding base well enough to call a genotype? | ≥ 94.5% |
| `freemix` | VerifyBAMID's estimate of the fraction of DNA that comes from *another person*. A mixture makes heterozygous sites look odd: allele fractions drift away from 0, 50 and 100% | ≤ 0.04 |
| `chrX_het_rate` | fraction of chrX variant sites (outside the pseudo-autosomal regions) that are heterozygous. One X copy cannot be heterozygous, so XY samples sit near 0 | XY ≈ 0, XX ≈ 0.6 |
| `chrY_depth_ratio` | reads on chrY relative to the autosomes. One Y copy against two copies of each autosome gives about 0.5 | XY ≈ 0.5, XX ≈ 0 |
| `array_concordance` | how often the exome genotypes agree with the SNP-array genotypes of the same participant. The same person agrees almost perfectly; two different people agree only by chance (about 0.6 here) | ≥ 0.95 |
| `PC1`–`PC4` | principal components: a few numbers summarising genome-wide genotype patterns. People with similar genetic ancestry sit close together. PCs describe genetic similarity, not identity or race | – |
| `pr_eur` | peddy's probability that the genotypes resemble the European reference group of the 1000 Genomes Project | – |
| `kinship` | KING kinship coefficient: about 0.5 for the same person or identical twins, 0.25 for parent–child or full siblings, 0.125 for second-degree and 0.0625 for third-degree relatives | – |

### The bookkeeping rule
Every removal goes through one function that records **which step** removed the sample and
**why**, and adds a line to the funnel. Nothing leaves the cohort without a written reason.
Run the cells in order; to redo a step, use *Kernel → Restart & Run All*.

*If a step will not run, catch up from the checkpoint and carry on, e.g. for step 4:*
`ck = pd.read_csv(CHECK / "practice2_excluded_samples.tsv", sep="\t")`, then
`remaining -= set(ck.sample_id[ck.step <= "4_identity"])`.

In [ ]:
qc = manifest.merge(metrics, on="sample_id", validate="one_to_one").set_index("sample_id").sort_index()
remaining = set(qc.index)
excluded = []      # one row per removed sample
flags = {}         # sample_id -> why it is kept but must be looked at
funnel = [{"step": "0_sequenced", "rule": "all delivered exomes", "n_removed": 0,
           "n_remaining": len(remaining)}]

def current():
    """The samples still in the analysis set."""
    return qc[qc.index.isin(remaining)]

def remove(reasons, step, rule):
    """reasons = {sample_id: why}. Remove them, record step and reason, add a funnel line."""
    if any(line["step"] == step for line in funnel):
        print(f"{step} was already applied. To redo it, restart the kernel and run all cells.")
        return
    for sample_id, why in reasons.items():
        remaining.discard(sample_id)
        excluded.append({"sample_id": sample_id, "step": step, "reason": why})
    funnel.append({"step": step, "rule": rule, "n_removed": len(reasons),
                   "n_remaining": len(remaining)})
    print(f"{step}: removed {len(reasons)}, {len(remaining)} remain")

def record_set(step, rule):
    """A funnel line that names an analysis set (nothing is removed here)."""
    if not any(line["step"] == step for line in funnel):
        funnel.append({"step": step, "rule": rule, "n_removed": 0, "n_remaining": len(remaining)})

print(len(remaining), "samples to start with")

---
## 2 · Consent first
Governance comes before statistics. A participant who withdrew consent is removed before anyone
looks at their data, so withdrawn samples appear in no later table, plot or count.

**Predict, then run.** Suppose their QC metrics are excellent. Does that change anything?

In [ ]:
c = current()
withdrawn = c[c.consent_status == "withdrawn"]
remove({s: "consent withdrawn: removed before any QC" for s in withdrawn.index},
       "1_consent", "consent_status == withdrawn")

It changes nothing. Notice that we did not print their metrics, and we will not.

---
## 3 · Contamination and coverage
Two data-quality rules from the paper: VerifyBAMID **FREEMIX > 4%** (the tube holds a mixture of
two people) and **< 94.5% of CCDS bases at ≥ 10×** (too much of the coding sequence is too thin
to genotype). Contamination goes first for a reason: a mixture inflates heterozygosity, which
would confuse the sex check in step 4.

**What changes:** two funnel lines. **How to check:** each removed sample breaks its rule.

In [ ]:
c = current()
contaminated = c[c.freemix > 0.04]
remove({s: f"freemix {v:.3f} > 0.04" for s, v in contaminated.freemix.items()},
       "2_contamination", "freemix > 0.04 (VerifyBAMID)")

c = current()
low_coverage = c[c.pct_ccds_10x < 94.5]
remove({s: f"pct_ccds_10x {v:.2f} < 94.5" for s, v in low_coverage.pct_ccds_10x.items()},
       "3_coverage", "pct_ccds_10x < 94.5")

qc.loc[list(contaminated.index) + list(low_coverage.index),
       ["reported_sex", "sequencing_batch", "mean_depth", "pct_ccds_10x", "freemix", "chrX_het_rate"]]

Look at the contaminated **male** samples: their `chrX_het_rate` is several times the usual male
value of about 0.02, because a second person's DNA adds extra alleles at chrX sites. That is
why contamination is checked before the sex check, not after.

Two of the three low-coverage samples come from batch **B3**. Keep that in mind for step 6.

---
## 4 · Is every sample who it claims to be?
*Concept slide: "Cohort QC: is every sample who it claims to be?" (slide 24).*

Two independent checks:
* **Sex check:** genetic sex from the data (is Y present? are there two X copies?) against the
  sex in the manifest.
* **Array concordance:** exome genotypes against the SNP array of the same participant.

Cross the two checks and each cell of the table has its own meaning:

| | array agrees (≥ 0.95) | array disagrees (< 0.95) |
|---|---|---|
| **sex concordant** | pass | identity not confirmed: a same-sex swap the sex check cannot see → **exclude** |
| **sex discordant** | the *record* is wrong, not the DNA → **correct the manifest, keep** | likely sample swap → **exclude** |
| **sex ambiguous** | unusual sex chromosomes (e.g. XXY-like) → **keep, flag for review; never call it a swap** | identity not confirmed → **exclude** |

In [ ]:
Y_PRESENT, X_TWO_COPIES = 0.20, 0.30      # teaching thresholds

def genetic_sex(x_het, y_ratio):
    y_present = y_ratio >= Y_PRESENT
    two_x = x_het >= X_TWO_COPIES
    if y_present and not two_x:
        return "XY"
    if two_x and not y_present:
        return "XX"
    return "ambiguous"

c = current().copy()
c["genetic_sex"] = [genetic_sex(x, y) for x, y in zip(c.chrX_het_rate, c.chrY_depth_ratio)]
expected = c.reported_sex.map({"F": "XX", "M": "XY"})
c["sex_check"] = np.where(c.genetic_sex == "ambiguous", "ambiguous",
                          np.where(c.genetic_sex == expected, "concordant", "discordant"))
c["array_ok"] = c.array_concordance >= 0.95
pd.crosstab(c.sex_check, c.array_ok)

**Predict, then run.** The next cell lists every sample outside the "concordant + array agrees"
cell. For each one: exclude, correct the record, or flag for review?

In [ ]:
look = c[(c.sex_check != "concordant") | ~c.array_ok]
look[["participant_id", "reported_sex", "chrX_het_rate", "chrY_depth_ratio", "genetic_sex",
      "array_concordance", "sex_check"]]

In [ ]:
reasons = {}
for s, r in look.iterrows():
    if not r.array_ok:
        if r.sex_check == "discordant":
            reasons[s] = (f"array_concordance {r.array_concordance:.3f} < 0.95 and reported "
                          f"{r.reported_sex} but genetic {r.genetic_sex}: likely swap")
        else:
            reasons[s] = (f"array_concordance {r.array_concordance:.3f} < 0.95 with {r.sex_check} "
                          f"sex: identity not confirmed")
    elif r.sex_check == "discordant":
        flags[s] = "metadata error: reported sex corrected"
    else:
        flags[s] = "sex chromosomes ambiguous: review"
remove(reasons, "4_identity", "array_concordance < 0.95")

# correct the record in a copy and tell the study team; never edit the source file silently
manifest_corrected = manifest.set_index("sample_id")
for s, flag in flags.items():
    if flag.startswith("metadata error"):
        manifest_corrected.loc[s, "reported_sex"] = {"XX": "F", "XY": "M"}[c.loc[s, "genetic_sex"]]
pd.Series(flags, name="flag").rename_axis("sample_id").to_frame()

### Checkpoint 4 · four different stories behind "something is off"
* **The swapped pair** (consecutive sample IDs, as if neighbours on a plate). Each is
  sex-discordant *and* array-discordant: the DNA in each tube belongs to someone else. Exclude
  both and ask the lab to trace the plate. With the full array data you could test whether each
  tube matches the *other* participant's array, which would confirm the swap.
* **The metadata error.** Sex-discordant, yet the exome matches the participant's own array at
  0.999. The DNA is right and the record is wrong: correct the manifest (in a copy, and tell the
  study team) and keep the sample.
* **The XXY-like sample.** Y present *and* two X copies, with array concordance 0.999, so it is
  who it claims to be. This is biology, not a lab error: flag it for review by the study team,
  never call it a swap, and never "fix" it to fit a box.
* **The same-sex discordance.** The sex check passes, the array check fails. A sex check only
  sees swaps between people of different sex; the array check sees them all.

---
## 5 · Duplicates and relatives
KING kinship guide:

| kinship | relationship |
|---|---|
| > 0.354 | the same person twice (duplicate) or monozygotic (identical) twins |
| 0.177 – 0.354 | first degree: parent–child, full siblings |
| 0.0884 – 0.177 | second degree: half-siblings, grandparent, aunt or uncle |
| 0.0442 – 0.0884 | third degree: first cousins |

Most association tests assume unrelated samples, so related pairs are broken up. The paper kept
no pair with kinship above **0.0884**, which leaves third-degree relatives in.

In [ ]:
def kin_degree(k):
    if k > 0.354:
        return "duplicate / MZ twin"
    if k > 0.177:
        return "1st degree"
    if k > 0.0884:
        return "2nd degree"
    return "3rd degree"

kin["degree"] = kin.kinship.apply(kin_degree)
live = kin[kin.sample_id_1.isin(remaining) & kin.sample_id_2.isin(remaining)]
print("pairs in the file:", len(kin), "| pairs among remaining samples:", len(live))
live.degree.value_counts()

### 5a · Duplicates first
A pair above 0.354 is the same genome twice. **Same `participant_id`:** a technical duplicate;
keep the copy with the better `pct_ccds_10x`. **Different `participant_id`:** identical twins, or
one person enrolled twice. Genetics cannot tell these apart, so keep one and flag the pair for the
study team (same age? same centre?).

In [ ]:
pct = qc.pct_ccds_10x
dups = live[live.kinship > 0.354]
reasons = {}
for r in dups.itertuples():
    a, b = r.sample_id_1, r.sample_id_2
    if a in reasons or b in reasons:
        continue
    lose, keep = (a, b) if pct[a] < pct[b] else (b, a)
    if qc.participant_id[a] == qc.participant_id[b]:
        reasons[lose] = f"duplicate of {keep} (same participant {qc.participant_id[a]}), lower pct_ccds_10x"
    else:
        reasons[lose] = (f"kinship {r.kinship:.3f} with {keep} (different participant): "
                         f"MZ twin or unrecorded duplicate, keep one, review")
        flags[keep] = "kept member of a twin/duplicate pair: review"
remove(reasons, "5_duplicates", "kinship > 0.354: keep the better sample")

shown = sorted(set(dups.sample_id_1) | set(dups.sample_id_2))
qc.loc[shown, ["participant_id", "reported_sex", "age_at_recruitment", "recruitment_centre",
               "sequencing_batch", "pct_ccds_10x"]]

### 5b · Relatives: remove as few people as possible
The paper used a greedy rule (the `ukbtools` R package): repeatedly remove the sample with the
**most** remaining close relatives. When two samples tie, we remove the one with the lower
`pct_ccds_10x` (our tie-break). In a trio this removes the child, who is related to both
parents, and keeps both parents: two people kept instead of one.

**Predict, then run.** In a family of three full siblings, how many are removed?

In [ ]:
close = live[(live.kinship > 0.0884) & live.sample_id_1.isin(remaining) & live.sample_id_2.isin(remaining)]
pairs = list(zip(close.sample_id_1, close.sample_id_2))
reasons = {}
while pairs:
    counts = pd.Series([s for pair in pairs for s in pair]).value_counts()
    most = counts[counts == counts.max()].index
    victim = sorted(most, key=lambda s: (pct[s], s))[0]      # tie -> lower pct_ccds_10x
    reasons[victim] = f"kinship > 0.0884 with {counts.max()} remaining sample(s)"
    pairs = [p for p in pairs if victim not in p]
remove(reasons, "6_relatedness", "kinship > 0.0884: greedy, most relatives first")

pan = current()
record_set("pan_ancestry_set", "QC-passed, unrelated, any ancestry")
left = kin[kin.sample_id_1.isin(remaining) & kin.sample_id_2.isin(remaining)]
print("pairs left among the remaining samples:", left.degree.value_counts().to_dict())
assert (left.kinship <= 0.0884).all()

Two of three siblings are removed: the first has two relatives left, then the remaining pair is
broken. The pairs left are all third degree, which the 0.0884 rule allows. One more detail to
notice: a duplicated participant who also has a sibling was counted **once**, because the
duplicate was resolved before relatedness.

---
## 6 · Ancestry and batch: structure you keep track of
*Concept slide: "What a cohort can and cannot tell you" (slide 25).*

Neither genetic ancestry nor sequencing batch is an error. Both are structure that can create
false associations if ignored, and the paper handled them differently:

* **Ancestry.** The main analyses used a genetically homogeneous European-ancestry set: peddy
  Pr(European) ≥ 0.99, then within ±4 SD of the mean on each of the top four PCs.
* **Batch.** The exomes arrived in four delivery batches. The authors made a "dummy phenotype" for
  each batch to find genes and variants that tracked batch, and excluded those genes and variants.

**What changes:** two funnel lines and the European analysis set. **How to check:** the samples
removed here are *not* QC failures; they remain in the pan-ancestry set.

In [ ]:
c = current()
not_eur = c[c.pr_eur < 0.99]
remove({s: f"pr_eur {v:.3f} < 0.99" for s, v in not_eur.pr_eur.items()},
       "7_ancestry_pr_eur", "pr_eur < 0.99 (peddy)")

eur_like = current()                                      # Pr(EUR) >= 0.99
PCS = ["PC1", "PC2", "PC3", "PC4"]
mean, sd = eur_like[PCS].mean(), eur_like[PCS].std()
z = (eur_like[PCS] - mean) / sd                           # distance from the mean, in SDs
furthest = z.abs().idxmax(axis=1)                         # the PC on which each sample is furthest out
outliers = {s: f"{pc} {z.loc[s, pc]:+.1f} SD from the mean (limit 4)"
            for s, pc in furthest.items() if abs(z.loc[s, pc]) > 4}
remove(outliers, "8_ancestry_pc_outlier", "outside mean +/- 4 SD on PC1-PC4")
record_set("eur_analysis_set", "European-ancestry analysis set")
eur = current()

In [ ]:
groups = [("European analysis set", pan.index.isin(eur.index), "#4C72B0", "o", 5),
          ("Pr(EUR) < 0.99", pan.index.isin(not_eur.index), "#DD8452", "o", 9),
          ("Pr(EUR) >= 0.99 but outside 4 SD", pan.index.isin(list(outliers)), "#C44E52", "x", 28)]
fig, axes = plt.subplots(1, 2, figsize=(10, 4.3))
for ax, (x, y) in zip(axes, [("PC1", "PC2"), ("PC3", "PC4")]):
    for label, mask, colour, marker, size in groups:
        ax.scatter(pan.loc[mask, x], pan.loc[mask, y], s=size, c=colour, marker=marker,
                   linewidths=1, alpha=0.75, label=f"{label} (n={mask.sum():,})")
    box = plt.Rectangle((mean[x] - 4 * sd[x], mean[y] - 4 * sd[y]), 8 * sd[x], 8 * sd[y],
                        fill=False, linestyle="--", edgecolor="0.3", linewidth=0.8)
    ax.add_patch(box)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
axes[0].set_title("PC1-PC2: the ancestry clusters", loc="left")
axes[1].set_title("PC3-PC4: the outliers only show here (dashed box = 4 SD)", loc="left")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.suptitle(f"Pan-ancestry analysis set, n = {len(pan):,}", fontsize=10)
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(OUT / "pca.png", dpi=150)
plt.show()

In [ ]:
batches = sorted(pan.sequencing_batch.unique())
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, column in zip(axes, ["mean_depth", "pct_ccds_10x"]):
    ax.boxplot([pan.loc[pan.sequencing_batch == b, column] for b in batches], widths=0.5)
    ax.set_xticks(range(1, len(batches) + 1))
    ax.set_xticklabels(batches)
    ax.set_xlabel("sequencing batch")
    ax.set_ylabel(column)
lc = qc.loc[list(low_coverage.index)]
axes[1].scatter([batches.index(b) + 1 for b in lc.sequencing_batch], lc.pct_ccds_10x, s=30,
                facecolors="none", edgecolors="#C44E52", label="excluded in step 3 (not in boxes)")
axes[1].axhline(94.5, color="0.3", linestyle="--", linewidth=0.8)
axes[1].annotate("94.5% rule", xy=(0.02, 94.5), xycoords=("axes fraction", "data"),
                 xytext=(0, 4), textcoords="offset points", fontsize=8)
axes[1].legend(loc="lower left")
axes[0].set_title("B3 has lower depth ...", loc="left")
axes[1].set_title("... and lower coverage, yet nearly all of it passes", loc="left")
fig.suptitle(f"Batch check on the pan-ancestry analysis set (n = {len(pan):,})", fontsize=10)
fig.tight_layout()
fig.savefig(OUT / "batch_ccds.png", dpi=150)
plt.show()

batch_median = pan.groupby("sequencing_batch")[["mean_depth", "pct_ccds_10x"]].median()
batch_median.round(2)

**Read the plot.** B3 sits about one percentage point lower on `pct_ccds_10x` and about ten reads
lower on mean depth, yet almost all of its samples pass the 94.5% line. Removing B3 would throw
away a quarter of the cohort for a difference that a model can handle. The usual response: keep
the samples, report the batch effect, include batch as a covariate, and check whether any gene or
variant result tracks batch, which is what the paper's per-batch dummy phenotypes did. Do not
move the 94.5% line to catch B3's weakest passing sample (94.81%): thresholds are set before you
look at the data, not after.

---
## The QC funnel
Every line of our funnel is one rule, and every number traces back to rows of the
excluded-samples table. For comparison, the paper's funnel (Wang et al. 2021, Methods):

| step | removed | remaining |
|---|---:|---:|
| exomes delivered | | 302,355 |
| sequencing-centre QC: sex discordance, contamination, unresolved duplicates, array discordance | flagged before delivery | |
| VerifyBAMID FREEMIX > 4% | 106 | |
| < 94.5% of CCDS bases at ≥ 10× | 5 | |
| KING kinship > 0.0884 (greedy, most relatives first) | 14,326 | **287,917** any ancestry |
| peddy Pr(European) < 0.99 | 18,212 | 269,706 |
| outside ±4 SD on the top four PCs | 535 | **269,171** European ancestry |

In [ ]:
funnel_df = pd.DataFrame(funnel)
excluded_df = pd.DataFrame(excluded).sort_values(["step", "sample_id"]).reset_index(drop=True)
funnel_df.to_csv(OUT / "practice2_qc_funnel.tsv", sep="\t", index=False)
excluded_df.to_csv(OUT / "practice2_excluded_samples.tsv", sep="\t", index=False)

# the funnel must add up, and every sample must be accounted for exactly once
n = funnel_df.n_remaining.to_numpy()
assert all(n[i] == n[i - 1] - funnel_df.n_removed[i] for i in range(1, len(n)))
assert len(excluded_df) + len(remaining) == len(qc) and excluded_df.sample_id.is_unique

for name in ["practice2_qc_funnel.tsv", "practice2_excluded_samples.tsv"]:
    mine = pd.read_csv(OUT / name, sep="\t")
    print(name, "matches checkpoint:", mine.equals(pd.read_csv(CHECK / name, sep="\t")))
funnel_df

**Check the paper's arithmetic yourself.** Predict first: should the numbers in the table above
add up exactly?

In [ ]:
print(f"302,355 - 106 - 5 - 14,326 = {302_355 - 106 - 5 - 14_326:,}   reported: 287,917")
print(f"287,917 - 18,212           = {287_917 - 18_212:,}   reported: 269,706")
print(f"269,706 - 535              = {269_706 - 535:,}   reported: 269,171")

Two of the three subtractions are off by one sample; the third is exact. The paper also says the
sequencing centre had already flagged samples before these steps, so the gap may sit in
bookkeeping we cannot see. The lesson is not that the paper is wrong. It is that a funnel kept by
hand drifts. Ours is produced by the same code that removes the samples, and the `assert` above
stops the notebook if it does not add up.

### Equity note: who is in "the cohort"?
In the paper, 269,171 of the 287,917 QC-passed exomes (93.5%) formed the European-ancestry set.
The authors also analysed 4,744 African, 1,475 East Asian and 5,714 South Asian ancestry
participants (Pr > 0.95 for that group) and ran a pan-ancestry analysis of 281,104 people. That
still leaves 6,813 QC-passed participants in none of the four groups, for example people whose
genotypes do not sit inside a single reference cluster.

Restricting to one ancestry makes a test statistically cleaner, and it leaves every finding least
tested in the people who were left out. In our cohort the `7_ancestry_pr_eur` rows of the
excluded table are not errors: they are participants, and an admixed participant fits no single
group at all. Report who was left out, analyse them where the numbers allow, and say what your
result cannot tell you.

In [ ]:
print(f"pan-ancestry set {len(pan):,} | European analysis set {len(eur):,} "
      f"({len(eur) / len(pan):.1%}) | left out of the European analysis: {len(pan) - len(eur):,}")

---
## Cohort card: fill it in
One line each, from your outputs above.

* **Delivered:** ___ samples from ___ participants
* **Governance:** ___ removed for withdrawn consent, before any QC
* **Data quality:** ___ contaminated (FREEMIX > 0.04), ___ low coverage (< 94.5% CCDS at 10×)
* **Identity:** ___ excluded (array discordance); ___ record corrected; ___ flagged for review
* **Duplicates and relatives:** ___ duplicates resolved; ___ relatives removed (kinship > 0.0884)
* **Pan-ancestry analysis set:** ___ samples
* **European analysis set:** ___ samples (___% of the pan-ancestry set); left out: ___
* **Batch:** B3 median `pct_ccds_10x` ___ vs ___ elsewhere; what you would do: ___
* **One thing this cohort cannot tell you:** ___

Then run the next cell and compare.

In [ ]:
removed = funnel_df.set_index("step").n_removed
print(f"delivered        {len(qc):,} samples from {qc.participant_id.nunique():,} participants")
print(f"governance       {removed['1_consent']} withdrawn")
print(f"data quality     {removed['2_contamination']} contaminated, {removed['3_coverage']} low coverage")
print(f"identity         {removed['4_identity']} excluded; flags kept for review:")
for s, flag in flags.items():
    print(f"                 {s}  {flag}")
print(f"duplicates       {removed['5_duplicates']} resolved; relatives removed {removed['6_relatedness']}")
print(f"pan-ancestry set {len(pan):,}")
print(f"European set     {len(eur):,} ({len(eur) / len(pan):.1%}); left out {len(pan) - len(eur):,}")
others = batch_median.pct_ccds_10x.drop("B3")
print(f"batch            B3 median pct_ccds_10x {batch_median.pct_ccds_10x['B3']:.2f} "
      f"vs {others.min():.2f}-{others.max():.2f} elsewhere")

---
### What you produced
| file | what it shows |
|---|---|
| `outputs/practice2_qc_funnel.tsv` | every rule, how many samples it removed, how many remain |
| `outputs/practice2_excluded_samples.tsv` | every removed sample with its step and reason |
| `outputs/pca.png` | the ancestry structure and the European restriction |
| `outputs/batch_ccds.png` | the B3 batch effect, and why it is not an exclusion |